# Cluster Sentences: blablabla

This notebook looks into techniques for defining clusters.
- First, sentences duplicates with a cosine similarity score of >=1 are removed.
- Compare resulting clusters of two lemmatisation methods (nltk, spacy, no lemma).

### Settings

In [ ]:
# Settings
embedding_file = "embedding_corpus_free_3600_250606.pickle"
similarity_file = "similarity_corpus_free_3600_250606.pickle"

raw_corpus_file = "corpus_free_3600_250606.csv"
used_st_model = "NeuML/pubmedbert-base-embeddings"

path_to_write_dict = '../../data/vectors/duplets'

# files for comparing lemmatisation methods
# Multiple vector files
folder = "explore_vectors"
filename1 = 'no_lemma_applied_df_xml2corpus_by_sentence_pmid_free_3600_250606.pickle'
filename2 = 'nltk_df_xml2corpus_by_sentence_pmid_free_3600_250606.pickle'
filename3 = 'spacy_df_xml2corpus_by_sentence_pmid_free_3600_250606.pickle'

## Initialisation

### Import and functions

In [ ]:
# import
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import torch
import seaborn as sns
from sentence_transformers import SentenceTransformer

### Functions

In [ ]:
def get_duplets_dataframe(similarity_matrix, threshold: float = 1.0):
    """Return a similarity dataframe that only contains duplets, where duplets are defined as sentences with a similarity score equal or higher than provided threshold."""
    # Substract 2 from the cosine similarity score that matched itself, so that it turns the value to -1.
    df_duplets = pd.DataFrame(
        np.subtract(
            similarity_matrix, 
            np.identity(similarity_matrix.shape[0]) *2
        )
    )

    # Drop columns and rows without duplets (defined by the threshold)
    df_duplets = df_duplets[df_duplets >= threshold].dropna(axis=0, how='all')
    df_duplets = df_duplets[df_duplets >= threshold].dropna(axis=1, how='all')

    # if df_duplets is empty (and therefore there are no duplets)
    if len(df_duplets.index) == 0:
        # raise an error
        raise ValueError("No duplets found in provided pd.DataFrame!")

    return df_duplets

def get_duplets_clusters(df_duplets):
    """Return a dictionary where the kept 'unique sentence index' (=key) references to the duplet index in a list (=values)."""
    # Create a dict with indexes on sentences with a duplet.
    potential_duplets = df_duplets.index
    duplet_dict = dict()

    # Iterate over potential duplets list
    # TODO: There should be a nicer way with groupby or something
    while len(potential_duplets) != 0:
        
        # Look at first item
        idx = potential_duplets[0]

        # For idx in list, find other idx that have same sentence.
        cluster = df_duplets.loc[idx, ~df_duplets.loc[idx].isna()].index

        # Store 'unique' idx in as key, and store 'duplet' idx as values
        duplet_dict[idx] = cluster.to_list()

        # Remove duplet index and first item from potential_duplets. (This will shorten the loop)
        potential_duplets = potential_duplets[~potential_duplets.isin(cluster)]
        potential_duplets = potential_duplets[1:]

    return duplet_dict

def delete_for_torch_tensor(x: torch.Tensor, row_exclude, axis: int = 0):
    """Return a new torch.Tensor with sub-arrays along an axis deleted.
    
    Parameters:
        x (torch.Tensor): Input tensor.

        row_exclude (list) : slice, array-like of ints or bools
        Indicate indices of sub-arrays to remove along the specified axis.

        axis (int): default = 0
        The axis along which to delete the subarray defined by row_exclude. 
    """
    # get indexes of interest (all indexes - indexes to remove)
    all_indexes = np.arange(x.shape[axis])
    indexes_of_interest = np.delete(all_indexes, row_exclude)

    # select rows and columns of interest (thus removing `indexes to remove`)
    if axis == 0:
        x = x[indexes_of_interest]
    if axis == 1:
        x = x[:, indexes_of_interest]
    
    return x

### Load

In [ ]:
# Load raw corpus
df_corpus = pd.read_csv(f'../../data/corpus/{raw_corpus_file}')

# Load cosine similarity scores
with open(f"../../data/vectors/{similarity_file}", 'rb') as handle:
    similarities = pickle.load(handle)

# Load sentence embeddings
with open(f"../../data/vectors/{embedding_file}", 'rb') as handle:
    embeddings = pickle.load(handle)


In [ ]:
# sample
similarities[:5, :5]

Similarity scores are correctly loaded if row and column indexes with the same number == `1`

In [ ]:
embeddings.shape

In [ ]:
similarities.shape

## Remove duplets (So that one unique remains)

As noted in notebook `sentence_pairs.ipynb`, there are multiple duplicated sentences. 
For clustering I would like to remove these duplicates, since we are interested in sentence similarities, I think that the 'high similarity' duplicate sentences have may influence the formation of clusters.

### Visualisation of duplets

In [ ]:
# Visualize clusters of duplets
sns.clustermap(get_duplets_dataframe(similarities).fillna(0))

### Remove duplets from `embeddings` and `similarities`

The removed duplets are registered in a dictionary that is written to `path_to_write_dict`, where the first occurance of the sentence is kept (=key in dict) and removed indexes are listed (as values in dict).

First occurrence is saved. The rest will be removed. (This will automatically shorten the list)
I cannot think of a reason why it would matter which sentence would be removed.

In [ ]:
# Get duplet dictionary
duplet_dict = get_duplets_clusters(
    get_duplets_dataframe(
        similarities
    )
)

# Create a list with indexes on sentences with a duplet.
duplet_to_remove = [item for layer in duplet_dict.values() for item in layer]
print(f"Number of indexes to remove: {len(duplet_to_remove)}")

# Drop duplets from `embeddings` (rows only)
print(f"Embedding shape with duplets: {embeddings.shape}")
embeddings = np.delete(embeddings, duplet_to_remove, 0)
print(f"Embedding shape without duplets: {embeddings.shape}")

# Drop duplets from `similarity` (both rows and columns)
print(f"Similarities shape with duplets: {similarities.shape}")
similarities = delete_for_torch_tensor(similarities, duplet_to_remove, axis=0)
similarities = delete_for_torch_tensor(similarities, duplet_to_remove, axis=1)
print(f"Similarities shape without duplets: {similarities.shape}")

In [ ]:
# Check if duplets are gone:
try:
    get_duplets_dataframe(
        similarities
    )
    print('We have a problem.. Duplets are still present..')

except ValueError:
    print("No duplets found in matrix.")

In [ ]:
# Save `duplet_dict` as .csv
# (This is important, because if the 'duplet' is a hit, we need to find it back in the corpus. Also if the hit occurs multiple times in the corpus.)

# Create `duplets` directory, if it does not yet exists.
Path(path_to_write_dict).mkdir(exist_ok = True)

# Save dict
with open(f"{path_to_write_dict}/dict_{'_'.join(embedding_file.split('_')[2:])}", "wb") as handle:
    pickle.dump(embeddings, handle)

## Continue